# Caishen Bank - Fraud Detection MVP: Ensemble Model Training

## Project Context
**Organization:** Caishen Bank (NYC-based International Bank)

**Mission:** Identify fraudulent activity within customer-facing bank accounts

**Objective:** Build an ensemble classifier (Random Forest or Gradient Boosting) MVP to detect fraudulent transactions

---

## Model Strategy
This notebook will:
1. Load preprocessed data
2. Train multiple ensemble classifiers
3. Compare model performance
4. Select and tune the best model
5. Evaluate final model performance
6. Save the production-ready model

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.model_selection import cross_val_score, GridSearchCV

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")
print(f"Random seed set to: {RANDOM_STATE}")

✓ Libraries imported successfully
Random seed set to: 42


## Step 1: Load Preprocessed Data

In [2]:
print("=== Loading Preprocessed Data ===")

# Load training data
X_train = pd.read_csv('../data/preprocessed/X_train.csv')
y_train = pd.read_csv('../data/preprocessed/y_train.csv').values.ravel()

# Load test data
X_test = pd.read_csv('../data/preprocessed/X_test.csv')
y_test = pd.read_csv('../data/preprocessed/y_test.csv').values.ravel()

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print(f"\nTraining labels distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for val, count in zip(unique, counts):
    print(f"  Class {val}: {count:,} ({count/len(y_train)*100:.4f}%)")

print(f"\nTest labels distribution:")
unique, counts = np.unique(y_test, return_counts=True)
for val, count in zip(unique, counts):
    print(f"  Class {val}: {count:,} ({count/len(y_test)*100:.4f}%)")

print(f"\nFeature names:")
print(X_train.columns.tolist())

print("\n✓ Data loaded successfully")

=== Loading Preprocessed Data ===

Training set: (5090096, 18)
Test set: (1272524, 18)

Training labels distribution:
  Class 0: 5,083,526 (99.8709%)
  Class 1: 6,570 (0.1291%)

Test labels distribution:
  Class 0: 1,270,881 (99.8709%)
  Class 1: 1,643 (0.1291%)

Feature names:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'balance_change_orig', 'balance_change_dest', 'amount_to_balance_ratio', 'is_zero_balance_orig', 'is_zero_balance_dest', 'balance_error_orig', 'balance_error_dest', 'has_balance_error', 'is_c2c_transaction', 'type_encoded', 'account_type_orig_encoded', 'account_type_dest_encoded']

✓ Data loaded successfully


## Step 2: Baseline Models - Random Forest vs Gradient Boosting

We'll train both ensemble methods with default parameters and `class_weight='balanced'` to handle class imbalance.

In [ ]:
print("=== Training Baseline Ensemble Models ===")
print("\nNote: Using class_weight='balanced' to handle severe class imbalance\n")

# Initialize models
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,  # Use all CPU cores
        verbose=1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        verbose=1
    )
}

# Store trained models and results
trained_models = {}
results = {}

# Train each model
for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}")
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    trained_models[name] = model
    results[name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    # Print results
    print(f"\n{name} Performance:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")

print("\n" + "="*60)
print("✓ Baseline models trained successfully")
print("="*60)

=== Training Baseline Ensemble Models ===

Note: Using class_weight='balanced' to handle severe class imbalance


Training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   52.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.3min finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.6s finished



Random Forest Performance:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    0.9976
  F1-Score:  0.9988
  ROC-AUC:   0.9988

Training Gradient Boosting...
      Iter       Train Loss   Remaining Time 
         1           0.0217           28.34m
         2           0.1620           28.09m
         3 519756041181949083715895300437075760627178465905353334921513380175656559340750805313519616.0000           79.64m
         4 513336654014868701316643351289655772157155935542003652682879470328289135955838874744258560.0000           65.97m
         5 513336654014868701316643351289655772157155935542003652682879470328289135955838874744258560.0000           57.67m
         6 513336654014868701316643351289655772157155935542003652682879470328289135955838874744258560.0000           52.21m
         7 513336654014868701316643351289655772157155935542003652682879470328289135955838874744258560.0000           48.08m
         8 51333665401486870131664335128965577215715593554200365268287947032828913595

## Step 3: Model Comparison

In [ ]:
print("=== Model Performance Comparison ===")

# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results.keys()],
    'Precision': [results[m]['precision'] for m in results.keys()],
    'Recall': [results[m]['recall'] for m in results.keys()],
    'F1-Score': [results[m]['f1_score'] for m in results.keys()],
    'ROC-AUC': [results[m]['roc_auc'] for m in results.keys()]
})

print("\n" + comparison_df.to_string(index=False))

# Identify best model based on F1-score (balances precision and recall)
best_model_name = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']
print(f"\n🏆 Best Model (by F1-Score): {best_model_name}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Metrics comparison
metrics_to_plot = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.35

for idx, model_name in enumerate(results.keys()):
    values = [comparison_df[comparison_df['Model'] == model_name][metric].values[0] 
              for metric in metrics_to_plot]
    axes[0].bar(x + idx*width, values, width, label=model_name, alpha=0.8)

axes[0].set_xlabel('Metrics', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Score', fontweight='bold', fontsize=12)
axes[0].set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
axes[0].set_xticks(x + width / 2)
axes[0].set_xticklabels(metrics_to_plot)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0, 1])

# Plot 2: ROC Curves
for model_name in results.keys():
    fpr, tpr, _ = roc_curve(y_test, results[model_name]['y_pred_proba'])
    auc = results[model_name]['roc_auc']
    axes[1].plot(fpr, tpr, label=f"{model_name} (AUC = {auc:.4f})", linewidth=2)

axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
axes[1].set_xlabel('False Positive Rate', fontweight='bold', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontweight='bold', fontsize=12)
axes[1].set_title('ROC Curves', fontweight='bold', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualizations complete")

## Step 4: Detailed Evaluation of Best Model

In [ ]:
print(f"=== Detailed Evaluation: {best_model_name} ===")

best_model = trained_models[best_model_name]
y_pred_best = results[best_model_name]['y_pred']
y_pred_proba_best = results[best_model_name]['y_pred_proba']

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Legitimate', 'Fraud']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives:  {cm[0,0]:,}")
print(f"False Positives: {cm[0,1]:,}")
print(f"False Negatives: {cm[1,0]:,}")
print(f"True Positives:  {cm[1,1]:,}")

# Calculate business metrics
total_frauds = cm[1,0] + cm[1,1]
detected_frauds = cm[1,1]
missed_frauds = cm[1,0]
false_alarms = cm[0,1]

print(f"\n📊 Business Metrics:")
print(f"  Total fraud cases: {total_frauds:,}")
print(f"  Frauds detected: {detected_frauds:,} ({detected_frauds/total_frauds*100:.2f}%)")
print(f"  Frauds missed: {missed_frauds:,} ({missed_frauds/total_frauds*100:.2f}%)")
print(f"  False alarms: {false_alarms:,}")

In [ ]:
# Visualize Confusion Matrix and PR Curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'],
            cbar_kws={'label': 'Count'})
axes[0].set_xlabel('Predicted Label', fontweight='bold', fontsize=12)
axes[0].set_ylabel('True Label', fontweight='bold', fontsize=12)
axes[0].set_title(f'Confusion Matrix - {best_model_name}', fontweight='bold', fontsize=14)

# Plot 2: Precision-Recall Curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba_best)
avg_precision = average_precision_score(y_test, y_pred_proba_best)

axes[1].plot(recall, precision, linewidth=2, label=f'AP = {avg_precision:.4f}')
axes[1].set_xlabel('Recall', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Precision', fontweight='bold', fontsize=12)
axes[1].set_title('Precision-Recall Curve', fontweight='bold', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("✓ Confusion matrix and PR curve plotted")

## Step 5: Feature Importance Analysis

In [ ]:
print(f"=== Feature Importance Analysis: {best_model_name} ===")

# Get feature importances
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_n = 15
top_features = feature_importance.head(top_n)

plt.barh(range(top_n), top_features['Importance'].values, color='steelblue', edgecolor='black')
plt.yticks(range(top_n), top_features['Feature'].values)
plt.xlabel('Importance', fontweight='bold', fontsize=12)
plt.ylabel('Feature', fontweight='bold', fontsize=12)
plt.title(f'Top {top_n} Feature Importances - {best_model_name}', fontweight='bold', fontsize=14)
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Feature importance analysis complete")

## Step 6: Save the Production Model

In [ ]:
print("=== Saving Production Model ===")

import os

# Use the best baseline model as final model
final_model = best_model
y_pred_final = y_pred_best
y_pred_proba_final = y_pred_proba_best

# Create models directory
os.makedirs('../models', exist_ok=True)

# Save the final model
model_filename = f'../models/fraud_detection_model_{best_model_name.lower().replace(" ", "_")}.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(final_model, f)

print(f"\n✓ Model saved: {model_filename}")

# Recalculate final confusion matrix for metadata
cm_final = confusion_matrix(y_test, y_pred_final)
total_frauds = cm_final[1,0] + cm_final[1,1]
detected_frauds = cm_final[1,1]
missed_frauds = cm_final[1,0]
false_alarms = cm_final[0,1]
roc_auc_final = roc_auc_score(y_test, y_pred_proba_final)

# Save model metadata
metadata = {
    'model_name': best_model_name,
    'features': list(X_train.columns),
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'performance_metrics': {
        'accuracy': float(accuracy_score(y_test, y_pred_final)),
        'precision': float(precision_score(y_test, y_pred_final)),
        'recall': float(recall_score(y_test, y_pred_final)),
        'f1_score': float(f1_score(y_test, y_pred_final)),
        'roc_auc': float(roc_auc_final)
    },
    'business_metrics': {
        'total_frauds': int(total_frauds),
        'detected_frauds': int(detected_frauds),
        'missed_frauds': int(missed_frauds),
        'false_alarms': int(false_alarms),
        'detection_rate': float(detected_frauds/total_frauds)
    }
}

metadata_filename = '../models/model_metadata.pkl'
with open(metadata_filename, 'wb') as f:
    pickle.dump(metadata, f)

print(f"✓ Metadata saved: {metadata_filename}")

print("\n" + "="*60)
print("All files saved successfully!")
print("="*60)

## Final Summary

In [ ]:
print("="*70)
print("CAISHEN BANK - FRAUD DETECTION MVP")
print("MODEL TRAINING COMPLETE")
print("="*70)

print("\n🎯 Project Objective:")
print("   Build ensemble classifier to detect fraudulent bank transactions")

print("\n🏆 Selected Model:")
print(f"   {best_model_name}")

print("\n📊 Model Performance:")
print(f"   Accuracy:  {metadata['performance_metrics']['accuracy']:.4f}")
print(f"   Precision: {metadata['performance_metrics']['precision']:.4f}")
print(f"   Recall:    {metadata['performance_metrics']['recall']:.4f}")
print(f"   F1-Score:  {metadata['performance_metrics']['f1_score']:.4f}")
print(f"   ROC-AUC:   {metadata['performance_metrics']['roc_auc']:.4f}")

print("\n💼 Business Impact:")
print(f"   Fraud detection rate: {metadata['business_metrics']['detection_rate']*100:.2f}%")
print(f"   Frauds caught: {metadata['business_metrics']['detected_frauds']:,} out of {metadata['business_metrics']['total_frauds']:,}")
print(f"   Frauds missed: {metadata['business_metrics']['missed_frauds']:,}")
print(f"   False alarms: {metadata['business_metrics']['false_alarms']:,}")

print("\n📁 Saved Files:")
print(f"   - {model_filename}")
print(f"   - {metadata_filename}")

print("\n✅ Next Steps:")
print("   1. Deploy model to production environment")
print("   2. Set up monitoring for model performance")
print("   3. Establish retraining schedule with new fraud data")
print("   4. Create API endpoint for real-time fraud detection")
print("   5. Integrate with Caishen Bank's transaction processing system")

print("\n" + "="*70)
print("MVP READY FOR DEPLOYMENT")
print("="*70)